# Council administration cleaning: Detect -> Judge -> Act
This notebook uses pandas and regex to inspect, clean and export only the council administration dataset. Run from the repository root. Input is preserved on first execution under `raw/before_cleaning/`; later runs replay that snapshot. No web requests or inferred demographic/contact values are introduced. The CDF datasets are not modified.


In [1]:
from pathlib import Path
import html
import re
import shutil
import pandas as pd

ROOT = Path.cwd()
filename = "db-unza26-csc4792-zimba_town_council_admin.csv"
target = ROOT / "data" / filename
snapshot = ROOT / "raw/before_cleaning" / filename
assert target.exists(), "Run from the repository root"
snapshot.parent.mkdir(parents=True, exist_ok=True)
if not snapshot.exists():
    shutil.copyfile(target, snapshot)
# Read as strings so phone prefixes and identifiers survive loading.
raw = pd.read_csv(snapshot, sep="|", dtype="string")
print("Input shape:", raw.shape)
print(raw.head().to_string(index=False))
raw.info()
print(raw.describe(include="all").to_string())
print("Missing values:\n", raw.isnull().sum())


Input shape: (2, 9)
  record_id name role_title department      phone                          email office_address                     source_page                                   source_url
ZTC-ADM-001 <NA>       <NA>       <NA>       <NA>        zimbacouncil@grz.gov.zm           <NA>      Zimba Town Council – Zimba             https://www.zimbacouncil.gov.zm/
ZTC-ADM-002 <NA>       <NA>       <NA> 0978080795 www.zimbatowncouncil@gmail.com           <NA> Contact Us – Zimba Town Council https://www.zimbacouncil.gov.zm/?page_id=275
<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   record_id       2 non-null      string
 1   name            0 non-null      string
 2   role_title      0 non-null      string
 3   department      0 non-null      string
 4   phone           1 non-null      string
 5   email           2 non-null      string
 6   office_address 

## Duplicates and text normalization: Detect
Inspect exact duplicates and repeated source URLs before making removal decisions. Decode HTML entities and collapse whitespace in ordinary text. Preserve URL punctuation, telephone prefixes and names; these are evidence fields, not a bag of words.


In [2]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Repeated source URLs:", raw.duplicated(subset=["source_url"]).sum())
clean = raw.copy()
def normalize(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"\s+", " ", html.unescape(str(value))).strip() or pd.NA
for column in clean.columns:
    if column.endswith("url"):
        clean[column] = clean[column].str.strip().replace("", pd.NA)
    else:
        clean[column] = clean[column].map(normalize).astype("string")


Exact duplicate rows: 0
Repeated source URLs: 0


## Duplicates: Judge and Act
The current file has two records from different contact pages. Remove exact duplicates excluding record IDs; only deduplicate by URL after confirming remaining rows on that URL have identical contents. Different officials or departments may share a page, so conflicting same-URL rows require review instead of silently keeping the first. This gives the same result as the existing administration cleaner on the current data while protecting future multi-record pages.


In [3]:
before = len(clean)
clean = clean.drop_duplicates(subset=[c for c in clean.columns if c != "record_id"], keep="first").copy()
assert not clean.duplicated(subset=["source_url"]).any(), "Review distinct records sharing a page"
clean = clean.drop_duplicates(subset=["source_url"], keep="first").copy()
duplicates_removed = before - len(clean)
print("Duplicates removed:", duplicates_removed)


Duplicates removed: 0


## Missing values: Detect -> Judge -> Act
This contact directory has no predefined prediction target. Names, role titles, departments, telephone numbers, emails and addresses are supporting features; do not invent a person's identity or discard a usable email-only contact point. Missing source URLs make a record unauditable and require review. Retain other gaps as blank cells. Keep phone values as strings, including the leading zero. Lowercase emails and normalize phone whitespace without guessing country codes or altering published digits.

IQR is not applicable: this file contains no numeric measurement columns. Phone numbers and record IDs are identifiers, not quantities; outliers corrected/removed are therefore not applicable rather than evidence of an IQR test.


In [4]:
print("Missing values:\n", clean.isnull().sum())
clean["email"] = clean["email"].str.lower()
clean["phone"] = clean["phone"].str.replace(r"\s+", " ", regex=True).str.strip()
email_pattern = r"[^\s@]+@[^\s@]+\.[^\s@]+"
invalid_email = clean["email"].notna() & ~clean["email"].str.fullmatch(email_pattern, na=False)
print("Email syntax flags:", int(invalid_email.sum()))
assert not invalid_email.any(), "Review email syntax without guessing a replacement"
print("IQR: not applicable; no numeric measurements")


Missing values:
 record_id         0
name              2
role_title        2
department        2
phone             1
email             0
office_address    2
source_page       0
source_url        0
dtype: int64
Email syntax flags: 0
IQR: not applicable; no numeric measurements


## Final validation and export
Preserve the original columns and source links. Optional punctuation removal/tokenization is omitted for this structured directory/profile: it would damage contacts and is unnecessary for the demographic measurements. Narrative notes remain intact. Export with `sep='|'` and `index=False`, retaining blanks, then reload as strings to check every exported field and schema. Existing identical files are not rewritten, allowing a file to remain open in a spreadsheet application.


In [5]:
assert clean["source_url"].notna().all(), "Review missing traceability URLs"
assert clean["source_url"].str.match(r"https?://", na=False).all()
assert clean["record_id"].notna().all() and clean["record_id"].is_unique
assert list(clean.columns) == list(raw.columns)
clean.info()
print(clean.describe(include="all").to_string())
exported = clean.to_csv(sep="|", index=False, na_rep="").encode("utf-8-sig")
if target.read_bytes() != exported:
    target.write_bytes(exported)
roundtrip = pd.read_csv(target, sep="|", dtype="string")
assert roundtrip.shape == clean.shape
pd.testing.assert_frame_equal(roundtrip.fillna(""), clean.astype("string").fillna(""), check_dtype=False)
print("Final rows:", len(clean), "columns:", len(clean.columns))
print("Duplicates removed:", duplicates_removed)
print("Final missing values:\n", clean.isnull().sum())
print("Saved:", filename)


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   record_id       2 non-null      string
 1   name            0 non-null      string
 2   role_title      0 non-null      string
 3   department      0 non-null      string
 4   phone           1 non-null      string
 5   email           2 non-null      string
 6   office_address  0 non-null      string
 7   source_page     2 non-null      string
 8   source_url      2 non-null      string
dtypes: string(9)
memory usage: 276.0 bytes
          record_id name role_title department       phone                    email office_address                 source_page                        source_url
count             2    0          0          0           1                        2              0                           2                                 2
unique            2    0          0          0           1   

## Handoff summary
The administration directory retains two rows and nine columns, with no duplicates removed. Missing names, roles, departments and addresses remain blank in both records, while one telephone number is missing; both records retain emails and source URLs. Text whitespace/entities and email case are normalized while telephone digits and prefixes remain unchanged. Numeric outlier analysis is not applicable to this contact directory.
